## Họ và tên: Huỳnh Quảng Tín
## MSSV: 24110353
Link github: https://github.com/AIVIETNAM-AIO-tinbmt79/Tri_tue_nhan_tao/blob/master/README.md 


In [7]:
%pip install pygame

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import sys
import random
from collections import deque
import heapq
import math

#### LOGIC ####

class Node:
    def __init__(self, state, parent, action, cost, name):
        self.state = state
        self.parent = parent
        self.action = action
        self.cost = cost
        self.name = name

def percept():

    def is_solvable(matrix):
        arr = []
        for row in matrix:
            for val in row:
                if val != 0:
                    arr.append(val)

        inversions = 0
        for i in range(len(arr)):
            for j in range(i + 1, len(arr)):

                if arr[i] > arr[j]:
                    inversions += 1
        return inversions % 2 == 0

    while True:

        list_num = list(range(9))

        random.shuffle(list_num)

        matrix = [
            list_num[i:i+3]
            for i in range(0, 9, 3)
        ]

        if is_solvable(matrix):
            return matrix

def interpret_input(matrix):
    for i in range(3):
        for j in range(3):
            if matrix[i][j] == 0:
                return (i, j)

def rules(state):
    x, y = state
    actions = []
    if x > 0: actions.append("UP")
    if x < 2: actions.append("DOWN")
    if y > 0: actions.append("LEFT")
    if y < 2: actions.append("RIGHT")
    return actions

def check_now(matrix):
    return 1 if matrix == [[1, 2, 3], [4, 5, 6], [7, 8, 0]] else 0

def action(chosen, matrix, state):
    x, y = state
    new_matrix = [row[:] for row in matrix]
    value1 = new_matrix[x][y]

    if chosen == "UP":
        new_matrix[x][y] = new_matrix[x-1][y]
        new_matrix[x-1][y] = value1
        return new_matrix, (x-1, y)
    elif chosen == "DOWN":
        new_matrix[x][y] = new_matrix[x+1][y]
        new_matrix[x+1][y] = value1
        return new_matrix, (x+1, y)
    elif chosen == "RIGHT":
        new_matrix[x][y] = new_matrix[x][y+1]
        new_matrix[x][y+1] = value1
        return new_matrix, (x, y+1)
    else:  # LEFT
        new_matrix[x][y] = new_matrix[x][y-1]
        new_matrix[x][y-1] = value1
        return new_matrix, (x, y-1)

def get_solution_path(goal_node_name, all_nodes):
    path = []
    current_name = goal_node_name
    while current_name is not None:
        node = all_nodes[current_name]
        path.append((node.state, node.action))
        current_name = node.parent
    path.reverse()
    return path

# BFS1: Kiem tra goal khi lay node ra khoi hang cho
def solve_8_puzzle_bfs1(initial_matrix):
    all_nodes = {}
    root_name = "A"
    root = Node(state=initial_matrix, parent=None, action=None, cost=0, name=root_name)
    all_nodes[root_name] = root
    queue = deque([root])
    reached = set()
    in_queue = set([str(root.state)])
    limit = 181440
    step = 0
    child_id = 1
    if check_now(initial_matrix) == 1:
        return get_solution_path(root_name, all_nodes)

    while len(queue) > 0 and step < limit:
        current_node = queue.popleft()
        in_queue.remove(str(current_node.state))
        if check_now(current_node.state) == 1:
            return get_solution_path(current_node.name, all_nodes)
        else:
            reached.add(str(current_node.state))

        zero_pos = interpret_input(current_node.state)
        valid_actions = rules(zero_pos)

        for act in valid_actions:
            new_matrix, _ = action(act, current_node.state, zero_pos)
            new_matrix_str = str(new_matrix)

            if new_matrix_str not in reached and new_matrix_str not in in_queue:
                child_name = f"Node_{child_id}"
                child_node = Node(new_matrix, current_node.name, act, current_node.cost + 1, child_name)
                queue.append(child_node)
                in_queue.add(new_matrix_str)
                all_nodes[child_name] = child_node
                child_id += 1
        step += 1
    return None

# BFS2: Kiem tra goal ngay khi tao child
def solve_8_puzzle_bfs2(initial_matrix):
    all_nodes = {}
    root_name = "A"
    root = Node(state=initial_matrix, parent=None, action=None, cost=0, name=root_name)
    all_nodes[root_name] = root

    if check_now(initial_matrix) == 1:
        return get_solution_path(root_name, all_nodes)

    queue = deque([root])
    reached = set()
    in_queue = set([str(root.state)])
    limit = 181440
    step = 0
    child_id = 1

    while len(queue) > 0 and step < limit:
        current_node = queue.popleft()
        in_queue.remove(str(current_node.state))
        zero_pos = interpret_input(current_node.state)
        valid_actions = rules(zero_pos)

        if check_now(current_node.state) == 1:
            return get_solution_path(current_node.name, all_nodes)
        else:
            reached.add(str(current_node.state))

        for act in valid_actions:
            new_matrix, _ = action(act, current_node.state, zero_pos)
            new_matrix_str = str(new_matrix)

            if new_matrix_str not in reached and new_matrix_str not in in_queue:
                child_name = f"Node_{child_id}"
                child_node = Node(new_matrix, current_node.name, act, current_node.cost + 1, child_name)

                if check_now(new_matrix) == 1:
                    all_nodes[child_name] = child_node
                    return get_solution_path(child_name, all_nodes)
                queue.append(child_node)
                in_queue.add(new_matrix_str)
                all_nodes[child_name] = child_node
                child_id += 1
        step += 1
    return None

# DFS2: Kiem tra goal ngay khi tao child
def solve_8_puzzle_dfs2(initial_matrix):
    all_nodes = {}
    root_name = "A"
    root = Node(state=initial_matrix, parent=None, action=None, cost=0, name=root_name)
    all_nodes[root_name] = root

    if check_now(initial_matrix) == 1:
        return get_solution_path(root_name, all_nodes)

    stack = deque([root])
    reached = set()
    in_stack = set([str(root.state)])
    limit = 181440
    step = 0
    child_id = 1

    while len(stack) > 0 and step < limit:
        current_node = stack.pop()
        in_stack.remove(str(current_node.state))
        zero_pos = interpret_input(current_node.state)
        valid_actions = rules(zero_pos)

        if check_now(current_node.state) == 1:
            return get_solution_path(current_node.name, all_nodes)
        else:
            reached.add(str(current_node.state))

        for act in valid_actions:
            new_matrix, _ = action(act, current_node.state, zero_pos)
            new_matrix_str = str(new_matrix)

            if new_matrix_str not in reached and new_matrix_str not in in_stack:
                child_name = f"Node_{child_id}"
                child_node = Node(new_matrix, current_node.name, act, current_node.cost + 1, child_name)

                if check_now(new_matrix) == 1:
                    all_nodes[child_name] = child_node
                    return get_solution_path(child_name, all_nodes)
                stack.append(child_node)
                in_stack.add(new_matrix_str)
                all_nodes[child_name] = child_node
                child_id += 1
        step += 1
    return None

# IDS: kiem tra goal ngay khi tao child
def solve_8_puzzle_ids(initial_matrix):
    all_nodes = {}
    child_id = [1]  

    def depth_limited_search(root_matrix, limit_depth):
        all_nodes.clear()
        child_id[0] = 1
        root_name = "A"
        root = Node(state=root_matrix, parent=None, action=None, cost=0, name=root_name)
        all_nodes[root_name] = root

        # Kiem tra goal tai trang thai goc
        if check_now(root_matrix) == 1:
            return get_solution_path(root_name, all_nodes), False  
        
        stack = deque([root])
        in_stack = set([str(root.state)])
        reached = set()
        result_cutoff = False

        while len(stack) > 0:
            current_node = stack.pop()
            in_stack.discard(str(current_node.state))

            if check_now(current_node.state) == 1:
                return get_solution_path(current_node.name, all_nodes), False

            reached.add(str(current_node.state))

            if current_node.cost >= limit_depth:
                result_cutoff = True
                continue

            zero_pos = interpret_input(current_node.state)
            valid_actions = rules(zero_pos)

            for act in valid_actions:
                new_matrix, _ = action(act, current_node.state, zero_pos)
                new_matrix_str = str(new_matrix)

                if new_matrix_str not in reached and new_matrix_str not in in_stack:
                    child_name = f"Node_{child_id[0]}"
                    child_node = Node(new_matrix, current_node.name, act, current_node.cost + 1, child_name)

                    # Kiem tra goal ngay khi tao child 
                    if check_now(new_matrix) == 1:
                        all_nodes[child_name] = child_node
                        return get_solution_path(child_name, all_nodes), False

                    stack.append(child_node)
                    in_stack.add(new_matrix_str)
                    all_nodes[child_name] = child_node
                    child_id[0] += 1

        return None, result_cutoff

    for depth in range(0, 181440):
        result, is_cutoff = depth_limited_search(initial_matrix, depth)
        if result is not None:
            return result
        if not is_cutoff:
            return None  # Khong co loi giai (khong bi cat, het node)

    return None

def solve_8_puzzle_ucs(initial_matrix):
    def calculate_cost(matrix):
        finish_matrix = [[1,2,3],[4,5,6],[7,8,0]]
        cost = 0
        for i in range(3):
            for j in range(3):
                if matrix[i][j] != 0 and matrix[i][j] != finish_matrix[i][j]:
                    cost += 1 
        return cost
    
    all_nodes = {}
    root_name = "A"
    
    cost_root = calculate_cost(initial_matrix) 
    root = Node(state=initial_matrix, parent=None, action=None, cost=cost_root, name=root_name)
    all_nodes[root_name] = root
    if check_now(initial_matrix) == 1:
        return get_solution_path(root_name, all_nodes)
    
    queue = []
    heapq.heappush(queue, (root.cost, 0, root))
    
    reached = {}
    limit = 181440
    step = 0
    child_id = 1
    
    while len(queue) > 0 and step < limit:
        _, _, current_node = heapq.heappop(queue)
        state_str = str(current_node.state)

        if state_str in reached and reached[state_str] <= current_node.cost:
            continue

        if check_now(current_node.state) == 1:
            return get_solution_path(current_node.name, all_nodes)
        
        reached[state_str] = current_node.cost  # lưu cost tốt nhất
        zero_pos = interpret_input(current_node.state)
        valid_actions = rules(zero_pos)

        for act in valid_actions:
            new_matrix, _ = action(act, current_node.state, zero_pos)
            new_matrix_str = str(new_matrix)
            new_cost = current_node.cost + calculate_cost(new_matrix)

            if new_matrix_str not in reached or reached[new_matrix_str] > new_cost:
                child_name = f"Node_{child_id}"
                child_node = Node(new_matrix, current_node.name, act, new_cost, child_name)
                heapq.heappush(queue, (child_node.cost, child_id, child_node))
                all_nodes[child_name] = child_node
                child_id += 1
        step += 1
    return None

def solve_8_puzzle_greedy(initial_matrix):
    #Số ô sai
    def calculate_cost(matrix):
        finish_matrix = [[1,2,3],[4,5,6],[7,8,0]]
        cost = 0
        for i in range(3):
            for j in range(3):
                if matrix[i][j] != 0 and matrix[i][j] != finish_matrix[i][j]:
                    cost += 1 
        return cost
    
    all_nodes = {}
    root_name = "A"
    
    cost_root = calculate_cost(initial_matrix) 
    root = Node(state=initial_matrix, parent=None, action=None, cost=cost_root, name=root_name)
    all_nodes[root_name] = root
    if check_now(initial_matrix) == 1:
        return get_solution_path(root_name, all_nodes)
    
    queue = []
    heapq.heappush(queue, (root.cost, 0, root))
    
    reached = set()
    limit = 181440
    step = 0
    child_id = 1
    
    while len(queue) > 0 and step < limit:
        _, _, current_node = heapq.heappop(queue)
        state_str = str(current_node.state)

        # Bỏ qua nếu đã xử lý với cost tốt hơn
        if state_str in reached:
            continue
        reached.add(state_str)

        if check_now(current_node.state) == 1:
            return get_solution_path(current_node.name, all_nodes)
        
        zero_pos = interpret_input(current_node.state)
        valid_actions = rules(zero_pos)

        for act in valid_actions:
            new_matrix, _ = action(act, current_node.state, zero_pos)
            new_matrix_str = str(new_matrix)
            cost = calculate_cost(new_matrix)

            if new_matrix_str not in reached:
                child_name = f"Node_{child_id}"
                child_node = Node(new_matrix, current_node.name, act, cost, child_name)
                heapq.heappush(queue, (child_node.cost, child_id, child_node))
                all_nodes[child_name] = child_node
                child_id += 1
        step += 1
    return None

def solve_8_puzzle_A_star(initial_matrix):
    def calculate_cost(matrix):
        cost = 0
        for i in range(3):
            for j in range(3):
                val = matrix[i][j]
                if val != 0:
                    target_i = (val - 1) // 3
                    target_j = (val - 1) % 3
                    cost += abs(i - target_i) + abs(j - target_j)
        return cost

    all_nodes = {}
    root_name = "A"

    h_root = calculate_cost(initial_matrix)
    g_root = 0
    root = Node(state=initial_matrix, parent=None, action=None, cost=g_root + h_root, name=root_name)
    root.g = g_root
    all_nodes[root_name] = root

    if check_now(initial_matrix) == 1:
        return get_solution_path(root_name, all_nodes)

    queue = []
    heapq.heappush(queue, (root.cost, 0, root))

    reached = {}       
    limit = 181440
    child_id = 1

    while len(queue) > 0 and len(reached) < limit:

        _, _, current_node = heapq.heappop(queue)

        state_str = str(current_node.state)
        #gặp trùng nếu cost nó lớn hơn = thì continue
        if state_str in reached and reached[state_str] <= current_node.g:
            continue

        reached[state_str] = current_node.g  

        if check_now(current_node.state) == 1:
            return get_solution_path(current_node.name, all_nodes)

        zero_pos = interpret_input(current_node.state)
        valid_actions = rules(zero_pos)

        for act in valid_actions:
            new_matrix, _ = action(act, current_node.state, zero_pos)
            new_matrix_str = str(new_matrix)

            g = current_node.g + 1
            if new_matrix_str not in reached or reached[new_matrix_str] > g:
                h = calculate_cost(new_matrix)
                f = g + h
                child_name = f"Node_{child_id}"
                child_node = Node(new_matrix, current_node.name, act, f, child_name)
                child_node.g = g
                heapq.heappush(queue, (child_node.cost, child_id, child_node))
                all_nodes[child_name] = child_node
                child_id += 1

    return None

def solve_8_puzzle_IDA_star(initial_matrix):
    def calculate_cost(matrix):
        cost = 0
        for i in range(3):
            for j in range(3):
                val = matrix[i][j]
                if val != 0:
                    target_i = (val - 1) // 3
                    target_j = (val - 1) % 3
                    cost += abs(i - target_i) + abs(j - target_j)
        return cost

    def search_f(current_node, f_limit):
        f = current_node.g + calculate_cost(current_node.state)
        # Nếu f vượt quá giới hạn, dừng nhánh này và trả về f để cập nhật limit mới
        if f > f_limit:
            return f, None
        
        if check_now(current_node.state) == 1:
            return "FOUND", current_node.name

        min_val = float('inf')
        zero_pos = interpret_input(current_node.state)
        valid_actions = rules(zero_pos)

        nonlocal child_id
        for act in valid_actions:
            new_matrix, _ = action(act, current_node.state, zero_pos)
            new_matrix_str = str(new_matrix)

            g = current_node.g + 1
            if new_matrix_str in reached and reached[new_matrix_str] <= g:
                continue

            reached[new_matrix_str] = g

            child_name = f"Node_{child_id}"
            child_node = Node(new_matrix, current_node.name, act, 0, child_name)
            child_node.g = g
            all_nodes[child_name] = child_node
            child_id += 1
            
            t, result_node_name = search_f(child_node, f_limit)
            
            if t == "FOUND":
                return "FOUND", result_node_name
            
            if t < min_val:
                min_val = t
            
            if new_matrix_str in reached:
                del reached[new_matrix_str]
                
        return min_val, None

    all_nodes = {}
    root_name = "A"
    
    h_root = calculate_cost(initial_matrix)
    root = Node(state=initial_matrix, parent=None, action=None, cost=h_root, name=root_name)
    root.g = 0
    all_nodes[root_name] = root
    reached = {str(initial_matrix): 0}
    
    f_limit = h_root
    child_id = 1
    
    while f_limit != float('inf'):
        t, target_name = search_f(root, f_limit)
        
        if t == "FOUND":
            return get_solution_path(target_name, all_nodes)

        if t == float('inf'):
            return None

        reached = {str(initial_matrix): 0}
        f_limit = t

    return None

def solve_8_puzzle_SHC(initial_matrix):
    def calculate_cost(matrix):
        cost = 0
        for i in range(3):
            for j in range(3):
                val = matrix[i][j]
                if val != 0:
                    target_i = (val - 1) // 3
                    target_j = (val - 1) % 3
                    cost += abs(i - target_i) + abs(j - target_j)
        return cost

    all_nodes = {}
    root_name = "A"

    cost_root = calculate_cost(initial_matrix)
    root = Node(state=initial_matrix, parent=None, action=None, cost=cost_root, name=root_name)
    all_nodes[root_name] = root

    current_node = root
    child_id = 1

    while True:
        zero_pos = interpret_input(current_node.state)
        valid_actions = rules(zero_pos)

        found_better = False

        for act in valid_actions:
            new_matrix, _ = action(act, current_node.state, zero_pos)
            next_cost = calculate_cost(new_matrix)

            if next_cost < current_node.cost:
                child_name = f"Node_{child_id}"
                child_node = Node(new_matrix, current_node.name, act, next_cost, child_name)
                all_nodes[child_name] = child_node
                child_id += 1

                current_node = child_node
                found_better = True
                break
        if not found_better:
            if check_now(current_node.state) == 1:
                return get_solution_path(current_node.name, all_nodes)
            return None  

def solve_8_puzzle_BHC(initial_matrix):
    def calculate_cost(matrix):
        cost = 0
        for i in range(3):
            for j in range(3):
                val = matrix[i][j]
                if val != 0:
                    target_i = (val - 1) // 3
                    target_j = (val - 1) % 3
                    cost += abs(i - target_i) + abs(j - target_j)
        return cost
    all_nodes = {}
    root_name = "A"

    cost_root = calculate_cost(initial_matrix)
    root = Node(state=initial_matrix, parent=None, action=None, cost=cost_root, name=root_name)
    all_nodes[root_name] = root

    current_node = root
    child_id = 1

    while True:
        zero_pos = interpret_input(current_node.state)
        valid_actions = rules(zero_pos)

        best_matrix = None
        best_act = None
        best_cost = current_node.cost
        found_better = False
        for act in valid_actions:
            new_matrix, _ = action(act, current_node.state, zero_pos)
            next_cost = calculate_cost(new_matrix)
            if next_cost < best_cost:
                best_cost = next_cost
                best_matrix = new_matrix
                best_act = act
                found_better = True

        if not found_better:
            if check_now(current_node.state) == 1:
                return get_solution_path(current_node.name, all_nodes)
            return None  

        child_name = f"Node_{child_id}"
        child_node = Node(best_matrix, current_node.name, best_act, best_cost, child_name)
        all_nodes[child_name] = child_node
        child_id += 1
        current_node = child_node


def solve_8_puzzle_SA(initial_matrix):
    def calculate_cost(matrix):
        cost = 0
        for i in range(3):
            for j in range(3):
                val = matrix[i][j]
                if val != 0:
                    target_i = (val - 1) // 3
                    target_j = (val - 1) % 3
                    cost += abs(i - target_i) + abs(j - target_j)
        return cost

    all_nodes = {}
    root_name = "A"
    cost_root = calculate_cost(initial_matrix)
    root = Node(state=initial_matrix,parent=None,action=None,cost=cost_root,name=root_name)
    all_nodes[root_name] = root
    current_node = root
    child_id = 1
    T0=1000
    Tmin=0.1
    alpha=0.95
    T = T0

    while T > Tmin:
        if check_now(current_node.state) == 1:
            return get_solution_path(current_node.name, all_nodes)

        zero_pos = interpret_input(current_node.state)
        valid_actions = rules(zero_pos)
        act = random.choice(valid_actions)

        new_matrix, _ = action(act,current_node.state,zero_pos)
        next_cost = calculate_cost(new_matrix)

        child_name = f"Node_{child_id}"
        child_node = Node(state=new_matrix,parent=current_node.name,action=act,cost=next_cost,name=child_name)
        all_nodes[child_name] = child_node
        child_id += 1

        delta = next_cost - current_node.cost

        if delta < 0:
            current_node = child_node

        else:
            p = math.exp(-delta / T)

            if random.random() < p:
                current_node = child_node

        T = alpha * T

    if check_now(current_node.state) == 1:
        return get_solution_path(current_node.name, all_nodes)

    return None


def solve_8_puzzle_hidden_initial(initial_matrix):
    all_nodes = {}
    root_name = "A"
    root = Node(state=initial_matrix, parent=None, action=None, cost=0, name=root_name)
    all_nodes[root_name] = root

    if all(check_now(matrix) for matrix in initial_matrix):
        return get_solution_path(root_name, all_nodes)

    queue = deque([root])
    reached = set()
    in_queue = set([str(root.state)])
    child_id = 1

    while len(queue) > 0:
        current_node = queue.popleft()
        in_queue.remove(str(current_node.state))
        current_beliefs = current_node.state
        reached.add(str(current_node.state))
        valid_actions = set()

        for matrix in current_beliefs:
            if check_now(matrix) != 1:
                zero_pos = interpret_input(matrix)
                valid_actions.update(rules(zero_pos))

        for act in valid_actions:
            new_beliefs = []
            for matrix in current_beliefs:
                if check_now(matrix):
                    new_beliefs.append(matrix)
                else:
                    zero_pos = interpret_input(matrix)

                    if act in rules(zero_pos):
                        new_matrix, _ = action(act, matrix, zero_pos)
                        new_beliefs.append(new_matrix)
                    else:
                        new_beliefs.append(matrix)

            new_state = new_beliefs

            if all(check_now(matrix) for matrix in new_beliefs):
                child_name = f"Node_{child_id}"
                child_node = Node(state=new_state, parent=current_node.name, action=act, cost=current_node.cost + 1, name=child_name)
                all_nodes[child_name] = child_node
                return get_solution_path(child_name, all_nodes)
        
            new_state_str = str(new_state)

            if new_state_str not in reached and new_state_str not in in_queue:
                child_name = f"Node_{child_id}"
                child_node = Node(state=new_state, parent=current_node.name, action=act, cost=current_node.cost + 1, name=child_name)
                queue.append(child_node)
                in_queue.add(new_state_str)
                all_nodes[child_name] = child_node
                child_id += 1

    return None


def solve_8_puzzle_hidden_goal(initial_matrix ,goal_list):
    all_nodes = {}
    root_name = "A"
    root = Node(state=initial_matrix, parent=None, action=None, cost=0, name=root_name)
    all_nodes[root_name] = root
    queue = deque([root])
    reached = set([str(root.state)])
    child_id = 1

    if any(root.state == goal for goal in goal_list):
        return get_solution_path(root_name, all_nodes)

    while len(queue) > 0:
        current_node = queue.popleft()
        zero_pos = interpret_input(current_node.state)
        valid_actions = rules(zero_pos)
    
        for act in valid_actions:
            new_matrix, _ = action(act, current_node.state, zero_pos)
            new_matrix_str = str(new_matrix)
        
            if new_matrix_str not in reached:
                child_name = f"Node_{child_id}"
                child_node = Node(state=new_matrix, parent=current_node.name, action=act, cost=current_node.cost + 1, name=child_name)
                all_nodes[child_name] = child_node

                if any(new_matrix == goal for goal in goal_list):
                    return get_solution_path(child_name, all_nodes)

                reached.add(new_matrix_str)
                queue.append(child_node)
                child_id += 1
    return None

from collections import deque
import random

def solve_8_puzzle_hidden_initial_part(initial_matrix):
    def is_solvable(matrix):

        arr = []

        for row in matrix:
            for val in row:
                if val != 0:
                    arr.append(val)
        inv = 0
        for i in range(len(arr)):
            for j in range(i + 1, len(arr)):
                if arr[i] > arr[j]:
                    inv += 1
        return inv % 2 == 0

    def generate_belief_matrix(observation):
        missing_numbers = []
        for num in range(9):
            found = False
            for row in observation:
                if num in row:
                    found = True
                    break
            if not found:
                missing_numbers.append(num)
        while True:
            shuffled = missing_numbers[:]
            random.shuffle(shuffled)
            idx = 0
            matrix = []
            for i in range(3):
                row = []
                for j in range(3):
                    if observation[i][j] == -1:
                        row.append(shuffled[idx])
                        idx += 1
                    else:
                        row.append(observation[i][j])

                matrix.append(row)

            if is_solvable(matrix):
                return matrix

    belief_states = []

    for observation in initial_matrix:
        belief_matrix = generate_belief_matrix(observation)
        while belief_matrix in belief_states:
            belief_matrix = generate_belief_matrix(observation)

        belief_states.append(belief_matrix)

    all_nodes = {}
    root_name = "A"
    root = Node(state=belief_states, parent=None, action=None, cost=0, name=root_name)
    all_nodes[root_name] = root

    if all(check_now(matrix) for matrix in belief_states):
        return get_solution_path(root_name, all_nodes)

    queue = deque([root])
    reached = set()
    in_queue = set([str(root.state)])
    child_id = 1

    while len(queue) > 0:
        current_node = queue.popleft()
        in_queue.remove(str(current_node.state))
        current_beliefs = current_node.state
        reached.add(str(current_node.state))
        valid_actions = set()

        for matrix in current_beliefs:
            if not check_now(matrix):
                zero_pos = interpret_input(matrix)
                valid_actions.update(rules(zero_pos))

        for act in valid_actions:
            new_beliefs = []
            for matrix in current_beliefs:
                if check_now(matrix):
                    new_beliefs.append(matrix)
                else:
                    zero_pos = interpret_input(matrix)

                    if act in rules(zero_pos):
                        new_matrix, _ = action(act, matrix, zero_pos)
                        new_beliefs.append(new_matrix)
                    else:
                        new_beliefs.append(matrix)

            new_state = new_beliefs

            if all(check_now(matrix) for matrix in new_beliefs):
                child_name = f"Node_{child_id}"
                child_node = Node(state=new_state, parent=current_node.name, action=act, cost=current_node.cost + 1, name=child_name)
                all_nodes[child_name] = child_node
                return get_solution_path(child_name, all_nodes)

            new_state_str = str(new_state)

            if new_state_str not in reached and new_state_str not in in_queue:
                child_name = f"Node_{child_id}"
                child_node = Node(state=new_state, parent=current_node.name, action=act, cost=current_node.cost + 1, name=child_name)
                queue.append(child_node)
                in_queue.add(new_state_str)
                all_nodes[child_name] = child_node
                child_id += 1

    return None

In [9]:
import pygame
import sys
import threading

WHITE       = (245, 245, 245)
BLACK       = (30,  30,  30)
DARK_BG     = (14,  16,  26)
TILE_CLR    = (64, 120, 210)
TILE_EMPTY  = (40,  44,  66)
GRAY        = (90,  95, 120)
ACCENT      = (255, 190,  60)

LOG_BG      = (18,  20,  34)
LOG_BORDER  = (50,  55,  85)
LOG_HEAD    = (40,  60, 120)
LOG_EVEN    = (24,  27,  44)
LOG_ODD     = (28,  32,  50)
LOG_TEXT    = (190, 205, 255)
LOG_ACT     = (255, 190,  60)

UNINFORMED_ALGOS = [
    {"key": "BFS1",   "label": "BFS 1",  "desc": "Breadth-First Search 1",   "color": (243, 156, 18)},
    {"key": "BFS2",   "label": "BFS 2",  "desc": "Breadth-First Search 2",   "color": (52,  152, 219)},
    {"key": "DFS2",   "label": "DFS 2",  "desc": "Depth-First Search",       "color": (46,  204, 113)},
    {"key": "IDS",    "label": "IDS",    "desc": "Iterative Deepening",      "color": (155,  89, 182)},
    {"key": "UCS",    "label": "UCS",    "desc": "Uniform-Cost Search",      "color": (230, 126,  34)},
]

INFORMED_ALGOS = [
    {"key": "GREEDY", "label": "Greedy", "desc": "Greedy Best-First Search", "color": (231,  76,  60)},
    {"key": "A*",     "label": "A*",     "desc": "A* Search",                "color": (240, 116,  90)},
    {"key": "IDA*",   "label": "IDA*",   "desc": "IDA* Search",              "color": (200, 100,  40)},
]

CLIMBING = [
    {"key": "SHC", "label": "SHC", "desc": "Simple Hill Climbing", "color": (204, 129, 210)},
    {"key": "BHC", "label": "BHC", "desc": "Best Hill Climbing",   "color": (232, 119, 230)},
]

ANNEALING = [
    {"key": "SA",  "label": "SA",  "desc": "Simulated Annealing",  "color": (255, 99, 132)},
]

HIDDEN_ENV = [
    {"key": "HIDDEN_GOAL",         "label": "H-Goal",    "desc": "Hidden Goal (BFS, multi-goal)",          "color": (80,  200, 200)},
    {"key": "HIDDEN_INIT",         "label": "H-Init",    "desc": "Hidden Initial (belief states)",         "color": (100, 180, 255)},
    {"key": "HIDDEN_INIT_PART",    "label": "H-Init P",  "desc": "Hidden Initial Partial Observe",         "color": (60,  140, 220)},
]

ALL_ALGOS = UNINFORMED_ALGOS + INFORMED_ALGOS + CLIMBING + ANNEALING + HIDDEN_ENV

ACTION_LABELS = {
    "UP":    "Di chuyen len",
    "DOWN":  "Di chuyen xuong",
    "LEFT":  "Di chuyen trai",
    "RIGHT": "Di chuyen phai",
    None:    "Trang thai ban dau",
}

HIDDEN_INIT_KEYS  = {"HIDDEN_INIT", "HIDDEN_INIT_PART"}
HIDDEN_GOAL_KEY   = "HIDDEN_GOAL"

def load_font(size, bold=False):
    candidates = ["segoeui", "arialuni", "notosans", "dejavusans", "freesans", "liberation sans", "tahoma"]
    for name in candidates:
        try:
            f = pygame.font.SysFont(name, size, bold=bold)
            if f.render("Test", True, (255,255,255)).get_width() > 10:
                return f
        except:
            pass
    return pygame.font.SysFont("arial", size, bold=bold)


def draw_board(screen, matrix, font, ox, oy, tile=88, margin=8):
    for i in range(3):
        for j in range(3):
            val = matrix[i][j]
            rx  = ox + j * (tile + margin)
            ry  = oy + i * (tile + margin)
            rect = pygame.Rect(rx, ry, tile, tile)
            if val == 0:
                pygame.draw.rect(screen, TILE_EMPTY, rect, border_radius=14)
                pygame.draw.rect(screen, (60, 65, 95), rect, 2, border_radius=14)
            else:
                shadow = pygame.Rect(rx + 3, ry + 5, tile, tile)
                pygame.draw.rect(screen, (20, 45, 110), shadow, border_radius=14)
                pygame.draw.rect(screen, TILE_CLR, rect, border_radius=14)
                hi = pygame.Rect(rx + 6, ry + 5, tile - 12, 10)
                pygame.draw.rect(screen, (100, 160, 255), hi, border_radius=6)
                txt = font.render(str(val), True, WHITE)
                screen.blit(txt, txt.get_rect(center=rect.center))


def draw_mini_board(screen, matrix, font, ox, oy, tile=34, margin=4, label="", font_lbl=None, highlight=False, obs=None):
    border_color = (255, 190, 60) if highlight else (50, 60, 100)
    total = 3 * tile + 2 * margin
    bg_rect = pygame.Rect(ox - 4, oy - 4 if not label else oy - 18, total + 8, total + 8 + (16 if label else 0))
    pygame.draw.rect(screen, (22, 26, 48), bg_rect, border_radius=8)
    pygame.draw.rect(screen, border_color, bg_rect, width=1, border_radius=8)

    if label and font_lbl:
        lbl_surf = font_lbl.render(label, True, (160, 175, 220))
        screen.blit(lbl_surf, (ox, oy - 16))

    for i in range(3):
        for j in range(3):
            val = matrix[i][j]
            rx  = ox + j * (tile + margin)
            ry  = oy + i * (tile + margin)
            rect = pygame.Rect(rx, ry, tile, tile)
            is_fixed = obs is not None and obs[i][j] != -1  
            if val == 0:
                pygame.draw.rect(screen, TILE_EMPTY, rect, border_radius=6)
            elif is_fixed:
                pygame.draw.rect(screen, (90, 140, 100), rect, border_radius=6)
                pygame.draw.rect(screen, (130, 200, 150), rect, width=2, border_radius=6)
                txt = font.render(str(val), True, WHITE)
                screen.blit(txt, txt.get_rect(center=rect.center))
            else:
                pygame.draw.rect(screen, (64, 120, 210), rect, border_radius=6)
                txt = font.render(str(val), True, WHITE)
                screen.blit(txt, txt.get_rect(center=rect.center))


def draw_button(screen, rect, label, font, style="default", hovered=False, pressed=False, disabled=False):
    offset = 0
    if disabled:
        bg, border, txt_c = (35, 38, 58), (55, 60, 90), GRAY
    elif style == "run":
        if pressed:      bg, border, txt_c = (30,160,80),   (60,200,120),  WHITE;         offset=2
        elif hovered:    bg, border, txt_c = (60,220,130),  (80,255,160),  (10,40,20)
        else:            bg, border, txt_c = (146,219,65),  (100,255,100), (10,40,20)
    elif style == "warning":
        if pressed:      bg, border, txt_c = (211,84,0),    (250,170,40),  WHITE;          offset=2
        elif hovered:    bg, border, txt_c = (250,170,40),  (255,200,80),  WHITE
        else:            bg, border, txt_c = (243,156,18),  (255,180,50),  WHITE
    elif style == "danger":
        if pressed:      bg, border, txt_c = (192,57,43),   (240,90,75),   WHITE;          offset=2
        elif hovered:    bg, border, txt_c = (240,90,75),   (250,110,90),  WHITE
        else:            bg, border, txt_c = (231,76,60),   (250,100,80),  WHITE
    else:
        bg, border, txt_c = TILE_CLR, (100,200,255), WHITE

    pygame.draw.rect(screen, bg,     rect, border_radius=9)
    pygame.draw.rect(screen, border, rect, width=2, border_radius=9)
    txt      = font.render(label, True, txt_c)
    txt_rect = txt.get_rect(center=rect.center)
    txt_rect.y += offset
    screen.blit(txt, txt_rect)


def draw_log_panel(screen, log_entries, scroll_offset, panel_rect, font_head, font_row):
    px, py, pw, ph = panel_rect
    pygame.draw.rect(screen, LOG_BG,     panel_rect, border_radius=12)
    pygame.draw.rect(screen, LOG_BORDER, panel_rect, width=1, border_radius=12)

    header_h = 34
    hdr_rect  = pygame.Rect(px, py, pw, header_h)
    pygame.draw.rect(screen, LOG_HEAD, hdr_rect, border_radius=12)
    pygame.draw.rect(screen, LOG_HEAD, pygame.Rect(px, py + header_h//2, pw, header_h//2))
    hdr_txt = font_head.render("Log cac buoc thuc hien", True, WHITE)
    screen.blit(hdr_txt, hdr_txt.get_rect(center=hdr_rect.center))

    row_h        = 28
    content_area = pygame.Rect(px, py + header_h, pw, ph - header_h)
    clip         = screen.get_clip()
    screen.set_clip(content_area)

    for idx, entry in enumerate(log_entries):
        row_y = py + header_h + idx * row_h - scroll_offset
        if row_y + row_h < py + header_h or row_y > py + ph:
            continue
        row_rect = pygame.Rect(px + 2, row_y, pw - 4, row_h - 2)
        bg = (28, 55, 38) if idx == len(log_entries)-1 else (LOG_EVEN if idx%2==0 else LOG_ODD)
        pygame.draw.rect(screen, bg, row_rect, border_radius=4)

        screen.blit(font_row.render(f"B{entry['step']:03d}", True, ACCENT),      (px+8,   row_y+6))
        screen.blit(font_row.render(entry['action_label'],   True, LOG_ACT),     (px+56,  row_y+6))
        rows_str = " | ".join(" ".join(str(v) for v in r) for r in entry['matrix'])
        screen.blit(font_row.render(rows_str,                True, LOG_TEXT),    (px+200, row_y+6))

    screen.set_clip(clip)

    total_h = len(log_entries) * row_h
    view_h  = ph - header_h
    if total_h > view_h:
        ratio = view_h / total_h
        bar_h = max(24, int(view_h * ratio))
        bar_y = py + header_h + int(scroll_offset / max(1, total_h) * view_h)
        pygame.draw.rect(screen, (70,80,120), pygame.Rect(px + pw - 9, bar_y, 6, bar_h), border_radius=3)


# ─────────────────────────────────────────────────────────────────
#  SCROLLABLE ALGO PANEL
# ─────────────────────────────────────────────────────────────────

ALGO_GROUPS = [
    {"label": "Tim kiem khong co thong tin:", "algos": UNINFORMED_ALGOS},
    {"label": "Tim kiem co thong tin:",        "algos": INFORMED_ALGOS},
    {"label": "Leo doi:",                      "algos": CLIMBING},
    {"label": "Mo phong luyen thep:",          "algos": ANNEALING},
    {"label": "Moi truong khong nhin thay:",   "algos": HIDDEN_ENV},
]


def build_algo_scroll_content(ctrl_w, algo_cols=3):
    btn_w   = (ctrl_w - 8 * (algo_cols - 1)) // algo_cols
    btn_h   = 35
    pad     = 8
    lbl_h   = 20
    grp_gap = 6
    items   = []
    cy      = 4

    for grp in ALGO_GROUPS:
        items.append({"type": "label", "text": grp["label"], "y": cy})
        cy += lbl_h

        algos = grp["algos"]
        rows  = (len(algos) + algo_cols - 1) // algo_cols
        for i, algo in enumerate(algos):
            col = i % algo_cols
            row = i // algo_cols
            ax  = col * (btn_w + pad)
            ay  = cy + row * (btn_h + pad)
            items.append({
                "type":  "btn",
                "key":   algo["key"],
                "label": algo["label"],
                "color": algo["color"],
                "y":     ay,
                "rect":  pygame.Rect(ax, ay, btn_w, btn_h),
            })
        cy += rows * (btn_h + pad) + grp_gap

    return cy + 4, items


def draw_algo_scroll_panel(screen, items, scroll_offset, panel_rect, selected_algo, mouse, mouse_pressed, is_animating, font_med, font_small):
    px, py, pw, ph = panel_rect
    pygame.draw.rect(screen, (18, 21, 38), pygame.Rect(px, py, pw, ph), border_radius=10)
    pygame.draw.rect(screen, (45, 52, 90), pygame.Rect(px, py, pw, ph), width=1, border_radius=10)

    clip_rect = pygame.Rect(px, py, pw, ph)
    old_clip  = screen.get_clip()
    screen.set_clip(clip_rect)

    for item in items:
        iy = py + item["y"] - scroll_offset
        if item["type"] == "label":
            if py <= iy <= py + ph:
                screen.blit(font_small.render(item["text"], True, (120, 135, 185)), (px + 6, iy))
        else:
            btn_rect = item["rect"].move(px + 6, py - scroll_offset)
            if btn_rect.bottom < py or btn_rect.top > py + ph:
                continue

            base_color = item["color"]
            is_sel     = item["key"] == selected_algo
            is_hov     = btn_rect.collidepoint(mouse) and not is_animating
            is_prs     = is_hov and mouse_pressed
            offset     = 0

            if is_sel:
                bg, border, txt_c = (20, 24, 38), base_color, base_color
            elif is_prs:
                bg = tuple(max(0, c - 40) for c in base_color)
                border, txt_c, offset = bg, WHITE, 2
            elif is_hov:
                bg = tuple(min(255, c + 30) for c in base_color)
                border, txt_c = bg, WHITE
            else:
                bg, border, txt_c = base_color, base_color, WHITE

            pygame.draw.rect(screen, bg,     btn_rect, border_radius=6)
            pygame.draw.rect(screen, border, btn_rect, width=2, border_radius=6)
            lbl_surf = font_med.render(item["label"], True, txt_c)
            lbl_r    = lbl_surf.get_rect(center=btn_rect.center)
            lbl_r.y += offset
            screen.blit(lbl_surf, lbl_r)

    screen.set_clip(old_clip)

    total_h  = max(1, sum((20 if i["type"] == "label" else i["rect"].height + 8) for i in items))
    if total_h > ph:
        ratio  = ph / total_h
        bar_h  = max(20, int(ph * ratio))
        bar_y  = py + int(scroll_offset / total_h * ph)
        pygame.draw.rect(screen, (70, 80, 120), pygame.Rect(px + pw - 7, bar_y, 5, bar_h), border_radius=3)


def hit_algo_in_scroll(items, scroll_offset, panel_rect, pos):
    px, py, pw, ph = panel_rect
    if not pygame.Rect(px, py, pw, ph).collidepoint(pos):
        return None
    for item in items:
        if item["type"] != "btn":
            continue
        btn_rect = item["rect"].move(px + 6, py - scroll_offset)
        if btn_rect.collidepoint(pos):
            return item["key"]
    return None


def build_layout(W, H):
    BOARD_X, BOARD_Y = 30, 45
    BOARD_SIZE = 3 * 88 + 2 * 8   

    CTRL_X  = BOARD_X + BOARD_SIZE + 30   
    CTRL_Y  = BOARD_Y
    CTRL_W  = 370

    ALGO_PANEL_H = min(320, H - CTRL_Y - 130)
    algo_panel   = (CTRL_X, CTRL_Y, CTRL_W, ALGO_PANEL_H)

    BTN_Y   = CTRL_Y + ALGO_PANEL_H + 10
    BTN_H   = 44
    BTN_GAP = 8
    BTN_W   = (CTRL_W - BTN_GAP * 2) // 3

    btn_run   = pygame.Rect(CTRL_X,                         BTN_Y, BTN_W, BTN_H)
    btn_new   = pygame.Rect(CTRL_X + BTN_W + BTN_GAP,       BTN_Y, BTN_W, BTN_H)
    btn_reset = pygame.Rect(CTRL_X + (BTN_W + BTN_GAP) * 2, BTN_Y, BTN_W, BTN_H)

    status_y  = BTN_Y + BTN_H + 10
    desc_y    = status_y + 22

    log_rect  = pygame.Rect(CTRL_X + CTRL_W + 20, 20, W - (CTRL_X + CTRL_W + 20) - 20, H - 40)

    MINI_H       = 3 * 38 + 2 * 5
    btn_goal_new = pygame.Rect(BOARD_X, BOARD_Y + BOARD_SIZE + 36 + MINI_H + 10, 130, 32)

    return {
        "BOARD_X":      BOARD_X,
        "BOARD_Y":      BOARD_Y,
        "CTRL_X":       CTRL_X,
        "CTRL_W":       CTRL_W,
        "algo_panel":   algo_panel,
        "btn_run":      btn_run,
        "btn_new":      btn_new,
        "btn_reset":    btn_reset,
        "btn_goal_new": btn_goal_new,
        "status_y":     status_y,
        "desc_y":       desc_y,
        "log_rect":     log_rect,
    }


def draw_side_boards(screen, selected_algo, board_x, board_y, belief_boards, goal_boards, font_tile, font_mini, font_lbl, btn_goal_new, mouse, mouse_pressed, is_animating, belief_obs=None):
    TILE  = 88
    MARG  = 8
    BSIZE = 3 * TILE + 2 * MARG   

    if selected_algo in HIDDEN_INIT_KEYS and belief_boards:
        lbl1 = font_lbl.render("Board 1 (trang thai 1):", True, (120, 135, 185))
        screen.blit(lbl1, (board_x, board_y - 20))
        b0 = belief_boards[0]
        if isinstance(b0, tuple): b0 = list(b0)
        draw_board(screen, b0, font_tile, board_x, board_y)

        sep_y = board_y + BSIZE + 14
        pygame.draw.line(screen, (50, 55, 90), (board_x, sep_y), (board_x + BSIZE, sep_y), 1)

        lbl2 = font_lbl.render("Board 2 (trang thai 2):", True, (120, 135, 185))
        board2_y = sep_y + 12
        screen.blit(lbl2, (board_x, board2_y))
        if len(belief_boards) >= 2:
            b1 = belief_boards[1]
            if isinstance(b1, tuple): b1 = list(b1)
            draw_board(screen, b1, font_tile, board_x, board2_y + 20)

        if selected_algo == "HIDDEN_INIT_PART" and belief_obs is not None:
            legend_y = board2_y + BSIZE + 40
            pygame.draw.rect(screen, (90, 140, 100), pygame.Rect(board_x, legend_y, 14, 14), border_radius=3)
            screen.blit(font_lbl.render(": O co dinh xung quanh 0", True, (150, 200, 160)), (board_x + 18, legend_y))
            pygame.draw.rect(screen, (64, 120, 210), pygame.Rect(board_x, legend_y + 22, 14, 14), border_radius=3)
            screen.blit(font_lbl.render(": O bi an ngau nhien", True, (150, 180, 220)), (board_x + 18, legend_y + 22))

    elif selected_algo == HIDDEN_GOAL_KEY and goal_boards:
        lbl = font_lbl.render("Goal states:", True, (120, 135, 185))
        screen.blit(lbl, (board_x, board_y + BSIZE + 18))

        MINI_TILE = 38
        MINI_MARG = 5
        MINI_SIZE = 3 * MINI_TILE + 2 * MINI_MARG
        goal_row_y = board_y + BSIZE + 36
        for idx, mat in enumerate(goal_boards[:2]):
            ox = board_x + idx * (MINI_SIZE + 16)
            draw_mini_board(screen, mat, font_mini, ox, goal_row_y, tile=MINI_TILE, margin=MINI_MARG, label=f"Goal {idx+1}", font_lbl=font_lbl, highlight=True)

        hov = btn_goal_new.collidepoint(mouse) and not is_animating
        prs = hov and mouse_pressed
        bg  = (50, 140, 160) if prs else (70, 190, 210) if hov else (45, 155, 175)
        pygame.draw.rect(screen, bg,              btn_goal_new, border_radius=7)
        pygame.draw.rect(screen, (100, 230, 255), btn_goal_new, width=2, border_radius=7)
        txt = font_lbl.render("Goal moi", True, WHITE)
        screen.blit(txt, txt.get_rect(center=btn_goal_new.center))


# ─────────────────────────────────────────────────────────────────
#  HÀM HELPER SINH MA TRẬN CHO H-INIT VÀ H-INIT-P (FIX CHẠY NHANH)
# ─────────────────────────────────────────────────────────────────

def generate_hidden_init_boards():
    """Sinh b2 từ b1 qua vài bước di chuyển ngẫu nhiên giúp H_init giải cực nhanh"""
    b1 = percept()
    def _move_random(base_mat, steps=2):
        import random as _r
        mat = [row[:] for row in base_mat]
        for _ in range(steps):
            r0, c0 = 0, 0
            for i in range(3):
                for j in range(3):
                    if mat[i][j] == 0: r0, c0 = i, j
            moves = []
            if r0 > 0: moves.append((-1, 0))
            if r0 < 2: moves.append((1, 0))
            if c0 > 0: moves.append((0, -1))
            if c0 < 2: moves.append((0, 1))
            dr, dc = _r.choice(moves)
            mat[r0][c0], mat[r0+dr][c0+dc] = mat[r0+dr][c0+dc], mat[r0][c0]
        return mat

    b2 = _move_random(b1, steps=2)
    while b2 == b1:
        b2 = _move_random(b1, steps=3)
    return [b1, b2], None


def generate_hidden_init_part_boards():
    """Sinh H_init_P cố định xung quanh ô số 0, tránh bẫy vòng lặp vô hạn"""
    import random as _r
    b1 = percept()
    r0, c0 = 0, 0
    for i in range(3):
        for j in range(3):
            if b1[i][j] == 0: r0, c0 = i, j

    obs = [[-1 for _ in range(3)] for _ in range(3)]
    for i in range(3):
        for j in range(3):
            if abs(i - r0) <= 1 and abs(j - c0) <= 1:
                obs[i][j] = b1[i][j]

    missing = [n for n in range(9) if not any(n in row for row in obs)]
    empty_pos = [(i, j) for i in range(3) for j in range(3) if obs[i][j] == -1]

    if len(empty_pos) <= 1:
        return [b1, [row[:] for row in b1]], obs

    b2 = None
    for _ in range(100):
        shuffled = missing[:]
        _r.shuffle(shuffled)
        test_mat = [[v for v in row] for row in obs]
        for idx, (pi, pj) in enumerate(empty_pos):
            test_mat[pi][pj] = shuffled[idx]
        arr = [v for row in test_mat for v in row if v != 0]
        inv = sum(1 for x in range(len(arr)) for y in range(x+1, len(arr)) if arr[x] > arr[y])
        if inv % 2 == 0 and test_mat != b1:
            b2 = test_mat
            break
    if b2 is None:
        b2 = [row[:] for row in b1]

    return [b1, b2], obs


# ─────────────────────────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────────────────────────

def main_gui():
    pygame.init()
    W, H = 1200, 720
    screen = pygame.display.set_mode((W, H), pygame.RESIZABLE)
    pygame.display.set_caption("8-Puzzle Solver")

    font_tile  = load_font(44, bold=True)
    font_big   = load_font(20, bold=True)
    font_med   = load_font(16, bold=True)
    font_small = load_font(14)
    font_log_h = load_font(14, bold=True)
    font_log_r = load_font(12)
    font_desc  = load_font(12)
    font_mini  = load_font(13, bold=True)
    clock      = pygame.time.Clock()

    initial_matrix = percept()
    current_matrix = [row[:] for row in initial_matrix]
    selected_algo  = "BFS1"
    status_msg     = "San sang! Chon thuat toan va nhan RUN."
    status_type    = "info"
    is_fullscreen  = False

    is_animating = False
    solution_path, anim_index, last_update = [], 0, 0
    DELAY = 650
    log_entries, log_scroll = [], 0

    algo_scroll, algo_panel_total, algo_items = 0, 0, []

    # Quản lý bộ nhớ trạng thái cho Môi trường ẩn
    belief_boards         = []   # Trạng thái hiện tại (dùng để chạy hoạt ảnh)
    initial_belief_boards = []   # Lưu giữ trạng thái gốc để "Quay lai"
    belief_obs            = None  # Mặt nạ hiển thị ô cố định
    goal_boards           = []   

    solve_result = [None]   
    solve_thread = None
    is_solving   = False    

    layout = build_layout(W, H)
    def rebuild_algo_items():
        nonlocal algo_items, algo_panel_total
        _, _, pw, _ = layout["algo_panel"]
        algo_panel_total, algo_items = build_algo_scroll_content(pw - 12)
    rebuild_algo_items()

    running = True
    while running:
        screen.fill(DARK_BG)
        mouse         = pygame.mouse.get_pos()
        mouse_pressed = pygame.mouse.get_pressed()[0]

        algo_panel   = layout["algo_panel"]
        BOARD_X      = layout["BOARD_X"]
        BOARD_Y      = layout["BOARD_Y"]
        CTRL_X       = layout["CTRL_X"]
        CTRL_W       = layout["CTRL_W"]
        btn_run      = layout["btn_run"]
        btn_new      = layout["btn_new"]
        btn_reset    = layout["btn_reset"]
        btn_goal_new = layout["btn_goal_new"]
        status_y     = layout["status_y"]
        desc_y       = layout["desc_y"]
        LOG_RECT     = layout["log_rect"]

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False

            if event.type == pygame.VIDEORESIZE:
                W, H   = event.w, event.h
                screen = pygame.display.set_mode((W, H), pygame.RESIZABLE)
                layout = build_layout(W, H)
                rebuild_algo_items()

            if event.type == pygame.MOUSEWHEEL:
                px, py, pw, ph = algo_panel
                if pygame.Rect(px, py, pw, ph).collidepoint(mouse):
                    max_scroll = max(0, algo_panel_total - ph)
                    algo_scroll = max(0, min(max_scroll, algo_scroll - event.y * 20))
                else:
                    log_scroll = max(0, log_scroll - event.y * 20)

            if event.type == pygame.MOUSEBUTTONDOWN and event.button == 1 and not is_animating and not is_solving:
                key = hit_algo_in_scroll(algo_items, algo_scroll, algo_panel, event.pos)
                if key:
                    selected_algo = key
                    belief_boards, initial_belief_boards, goal_boards, belief_obs = [], [], [], None

                    if selected_algo == "HIDDEN_INIT":
                        belief_boards, belief_obs = generate_hidden_init_boards()
                        initial_belief_boards = [row[:] for row in belief_boards]
                    elif selected_algo == "HIDDEN_INIT_PART":
                        belief_boards, belief_obs = generate_hidden_init_part_boards()
                        initial_belief_boards = [row[:] for row in belief_boards]
                    elif selected_algo == HIDDEN_GOAL_KEY:
                        goal_boards = [[[1,2,3],[4,5,6],[7,8,0]], [[1,2,3],[4,5,0],[7,8,6]]]

                # ─────────────────────────────────────────────────────────────
                # XỬ LÝ NÚT TRẠNG THÁI MỚI (NEW)
                # ─────────────────────────────────────────────────────────────
                if btn_new.collidepoint(event.pos):
                    solution_path, log_entries, log_scroll = [], [], 0
                    status_msg, status_type = "Trang thai moi! Nhan RUN de giai.", "info"
                    
                    if selected_algo == "HIDDEN_INIT":
                        belief_boards, belief_obs = generate_hidden_init_boards()
                        initial_belief_boards = [row[:] for row in belief_boards]
                    elif selected_algo == "HIDDEN_INIT_PART":
                        belief_boards, belief_obs = generate_hidden_init_part_boards()
                        initial_belief_boards = [row[:] for row in belief_boards]
                    else:
                        initial_matrix = percept()
                        current_matrix = [row[:] for row in initial_matrix]

                # ─────────────────────────────────────────────────────────────
                # XỬ LÝ NÚT QUAY LẠI (RESET)
                # ─────────────────────────────────────────────────────────────
                elif btn_reset.collidepoint(event.pos):
                    solution_path, log_entries, log_scroll = [], [], 0
                    status_msg, status_type = "Da quay lai trang thai ban dau.", "info"
                    
                    if selected_algo in HIDDEN_INIT_KEYS and initial_belief_boards:
                        belief_boards = [row[:] for row in initial_belief_boards]
                    else:
                        current_matrix = [row[:] for row in initial_matrix]

                elif btn_goal_new.collidepoint(event.pos) and selected_algo == HIDDEN_GOAL_KEY:
                    goal_boards = [percept(), percept()]
                    status_msg, status_type = "Da tao moi Goal states!", "info"

                elif btn_run.collidepoint(event.pos):
                    log_entries, log_scroll = [], 0
                    solve_result[0] = None
                    status_msg, status_type = f"Dang tinh toan ({selected_algo})...", "info"

                    _algo   = selected_algo
                    _mat    = [row[:] for row in current_matrix]
                    _goals  = [g for g in goal_boards] if goal_boards else [[[1,2,3],[4,5,6],[7,8,0]]]
                    _boards = [b for b in belief_boards] if belief_boards else []

                    def _solve():
                        res = None
                        try:
                            if   _algo == "BFS1":    res = solve_8_puzzle_bfs1(_mat)
                            elif _algo == "BFS2":    res = solve_8_puzzle_bfs2(_mat)
                            elif _algo == "DFS2":    res = solve_8_puzzle_dfs2(_mat)
                            elif _algo == "IDS":     res = solve_8_puzzle_ids(_mat)
                            elif _algo == "UCS":     res = solve_8_puzzle_ucs(_mat)
                            elif _algo == "GREEDY":  res = solve_8_puzzle_greedy(_mat)
                            elif _algo == "A*":      res = solve_8_puzzle_A_star(_mat)
                            elif _algo == "IDA*":    res = solve_8_puzzle_IDA_star(_mat)
                            elif _algo == "SHC":     res = solve_8_puzzle_SHC(_mat)
                            elif _algo == "BHC":     res = solve_8_puzzle_BHC(_mat)
                            elif _algo == "SA":      res = solve_8_puzzle_SA(_mat)
                            elif _algo == "HIDDEN_GOAL": res = solve_8_puzzle_hidden_goal(_mat, _goals)
                            elif _algo == "HIDDEN_INIT": res = solve_8_puzzle_hidden_initial(_boards)
                            elif _algo == "HIDDEN_INIT_PART": res = solve_8_puzzle_hidden_initial_part(_boards)
                        except NameError: pass
                        solve_result[0] = res if res is not None else "NONE"

                    is_solving  = True
                    solve_thread = threading.Thread(target=_solve, daemon=True)
                    solve_thread.start()

        if is_solving and solve_result[0] is not None:
            is_solving = False
            result = solve_result[0]
            if result == "NONE" or result is None:
                status_msg, status_type = "Khong tim thay duong di!", "error"
            else:
                solution_path = result
                is_animating  = True
                anim_index    = 0
                last_update   = pygame.time.get_ticks()
                status_msg    = f"Tim thay! {len(solution_path)-1} buoc ({selected_algo})"
                status_type   = "success"

        if is_animating:
            now = pygame.time.get_ticks()
            if now - last_update > DELAY:
                state, act = solution_path[anim_index]
                label = ACTION_LABELS.get(act, act if act else "?")

                if selected_algo in HIDDEN_INIT_KEYS:
                    boards = state if isinstance(state[0][0], list) else [state]
                    belief_boards = [list(b) for b in boards]
                    for bi, b in enumerate(boards):
                        log_entries.append({"step": anim_index * len(boards) + bi, "action_label": f"{label} (B{bi+1})", "matrix": b})
                else:
                    current_matrix = state
                    log_entries.append({"step": anim_index, "action_label": label, "matrix": state})

                total_h = len(log_entries) * 28
                view_h  = LOG_RECT[3] - 34
                if total_h > view_h: log_scroll = total_h - view_h
                anim_index += 1
                last_update = now
                if anim_index >= len(solution_path):
                    is_animating = False
                    status_msg, status_type = "HOAN THANH!", "success"

        # ===================== RENDERING GIAO DIỆN =====================
        if selected_algo in HIDDEN_INIT_KEYS:
            draw_side_boards(screen, selected_algo, BOARD_X, BOARD_Y, belief_boards, goal_boards, font_tile, font_mini, font_small, btn_goal_new, mouse, mouse_pressed, is_animating, belief_obs=belief_obs)
        else:
            lbl = font_small.render("Trang thai hien tai", True, (120,135,185))
            screen.blit(lbl, (BOARD_X, BOARD_Y - 25))
            draw_board(screen, current_matrix, font_tile, BOARD_X, BOARD_Y)
            draw_side_boards(screen, selected_algo, BOARD_X, BOARD_Y, belief_boards, goal_boards, font_tile, font_mini, font_small, btn_goal_new, mouse, mouse_pressed, is_animating, belief_obs=belief_obs)

        draw_algo_scroll_panel(screen, algo_items, algo_scroll, algo_panel, selected_algo, mouse, mouse_pressed, is_animating, font_med, font_small)

        for algo in ALL_ALGOS:
            if algo["key"] == selected_algo:
                screen.blit(font_desc.render(algo["desc"], True, (130,150,200)), (CTRL_X, desc_y))
                break

        draw_button(screen, btn_run,   "RUN",           font_big,   style="run", hovered=btn_run.collidepoint(mouse) and not is_animating, pressed=btn_run.collidepoint(mouse) and mouse_pressed and not is_animating, disabled=is_animating or is_solving)
        draw_button(screen, btn_new,   "Trang thai moi", font_small, style="warning", hovered=btn_new.collidepoint(mouse) and not is_animating, pressed=btn_new.collidepoint(mouse) and mouse_pressed and not is_animating, disabled=is_animating or is_solving)
        draw_button(screen, btn_reset, "Quay lai",       font_small, style="danger", hovered=btn_reset.collidepoint(mouse) and not is_animating, pressed=btn_reset.collidepoint(mouse) and mouse_pressed and not is_animating, disabled=is_animating or is_solving)

        s_color = (46,204,113) if status_type=="success" else (220,70,70) if status_type=="error" else (160,175,220)
        st_bg   = pygame.Rect(CTRL_X-4, status_y-4, CTRL_W+8, 26)
        pygame.draw.rect(screen, (24,28,48), st_bg, border_radius=6)
        screen.blit(font_small.render(status_msg, True, s_color), (CTRL_X, status_y))

        if is_solving:
            tick = (pygame.time.get_ticks() // 300) % 4
            screen.blit(font_small.render(f"Dang tinh toan{'.'*(tick+1)}", True, (100, 200, 255)), (CTRL_X, status_y + 24))
        elif is_animating:
            tick = (pygame.time.get_ticks() // 400) % 4
            screen.blit(font_small.render(f"Dang chay{'.'*(tick+1)}", True, ACCENT), (CTRL_X, status_y + 24))

        screen.blit(font_desc.render("F11: Phong to / Thu nho  |  Scroll chuot tren panel de cuon", True, (60,70,100)), (BOARD_X, H - 22))
        draw_log_panel(screen, log_entries, log_scroll, LOG_RECT, font_log_h, font_log_r)

        pygame.display.flip()
        clock.tick(60)

    pygame.quit()
    sys.exit()

if __name__ == "__main__":
    main_gui()

SystemExit: 